In [ ]:
!pip install -q transformers==4.43.0 tokenizers==0.19.1 einops timm torchvision pillow matplotlib

In [ ]:
import torch
import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers.dynamic_module_utils as dyn_utils
from unittest.mock import patch

# 🚨 [두더지 1번 완벽 제압] flash_attn 요구사항 몰래 삭제하기
orig_get_imports = dyn_utils.get_imports

def custom_get_imports(filename):
    imports = orig_get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports
# ==========================================================

model_id = "microsoft/Florence-2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ {device} 환경에서 Florence-2 모델을 불러오는 중...")

# 🚨 [두더지 2번 완벽 제압] 안정적인 구버전 환경 + 수문장 패치 동시 적용!
with patch("transformers.dynamic_module_utils.get_imports", custom_get_imports):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to(device)

print("✅ 모델 로딩 완벽 성공!\n")

# 2. 테스트용 거실 사진 불러오기
image_path = "/content/image(7).png"
image = Image.open(image_path).convert("RGB")

# 3. 사용자 입력받기 (예: chair, lamp, table)
user_input = input("🤖 찾고 싶은 물건을 영어로 입력하세요 (예: chair): ")

# 4. 프롬프트 및 추론
task_prompt = "<CAPTION_TO_PHRASE_GROUNDING>"
text_input = task_prompt + user_input

inputs = processor(text=text_input, images=image, return_tensors="pt").to(device)

print(f"\n🔍 '{user_input}'(을)를 찾는 중...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )

# 5. 결과 파싱 및 시각화
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=image.size)

fig, ax = plt.subplots(1, figsize=(10, 8))
ax.imshow(image)

results = parsed_answer.get(task_prompt, {})
bboxes = results.get('bboxes', [])
labels = results.get('labels', [])

if not bboxes:
    print(f"❌ 화면에서 '{user_input}'(을)를 찾을 수 없습니다.")
else:
    print(f"🎯 총 {len(bboxes)}개의 '{user_input}'(을)를 찾았습니다!\n")

    for box, label in zip(bboxes, labels):
        x1, y1, x2, y2 = box
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        print(f"[{label}] 바운딩 박스: ({x1:.1f}, {y1:.1f}) ~ ({x2:.1f}, {y2:.1f})")
        print(f" └─> 🤖 로봇 목표 좌표(Pick): X={center_x:.1f}, Y={center_y:.1f}\n")

        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.plot(center_x, center_y, 'ro')
        ax.text(x1, y1 - 10, label, color='red', fontsize=12, weight='bold', backgroundcolor='white')

plt.axis('off')
plt.show()

In [ ]:
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers.dynamic_module_utils as dyn_utils
from unittest.mock import patch

# 🚨 [수문장 무력화 패치]
orig_get_imports = dyn_utils.get_imports
def custom_get_imports(filename):
    imports = orig_get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports
# ==========================================================

model_id = "microsoft/Florence-2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ {device} 환경에서 Florence-2 모델을 불러오는 중...")

with patch("transformers.dynamic_module_utils.get_imports", custom_get_imports):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to(device)

print("✅ 모델 로딩 완벽 성공!\n")

# 2. 테스트용 사진 불러오기 (경로를 알맞게 수정하세요!)
image_path = "/content/image(7).png"
image = Image.open(image_path).convert("RGB")

# 3. [핵심 변경] 프롬프트를 '상세 묘사 모드'로 변경
task_prompt = "<MORE_DETAILED_CAPTION>"

# 묘사 모드에서는 뒤에 내가 찾는 단어를 붙이지 않고, 프롬프트 자체만 텍스트로 넘깁니다.
inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(device)

print("\n🔍 사진을 뚫어져라 분석하며 상황을 추론하는 중...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )

# 4. 결과 파싱
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=image.size)

# 5. [핵심 변경] 바운딩 박스를 그리지 않고, 추론된 문장만 깔끔하게 출력합니다.
print("\n========================================")
print("🧠 Florence-2의 상세 묘사(추론) 결과:")
print("========================================")
print(parsed_answer[task_prompt])
print("========================================\n")

# 사진은 참고용으로 화면에 띄워만 줍니다.
fig, ax = plt.subplots(1, figsize=(8, 6))
ax.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
# ==========================================
# 0단계: 필수 라이브러리 Import 및 패치 적용
# ==========================================
import torch
import time  # ⏱ 시간 측정용 라이브러리
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, AutoTokenizer
import transformers.dynamic_module_utils as dyn_utils
from unittest.mock import patch
from google.colab import userdata

# 1. 신분증 (Hugging Face Token) 확인 - Gemma 2는 토큰이 필수입니다!
clean_token = userdata.get('HF_TOKEN').strip()

# 🚨 [수문장 무력화 패치] Florence-2의 flash_attn 에러 우회
orig_get_imports = dyn_utils.get_imports
def custom_get_imports(filename):
    imports = orig_get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1단계: 모델 로딩 (Florence-2 & Gemma-2 2B)
# ==========================================
print("⏳ [시스템 초기화] 시각 센서(Florence-2)와 중앙 통제 AI(Gemma-2)를 가동합니다...\n")

# 1-1. Florence-2 (눈) 로딩
model_id = "microsoft/Florence-2-base"
with patch("transformers.dynamic_module_utils.get_imports", custom_get_imports):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    vision_model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to(device)

# 1-2. Gemma-2 2B (뇌) 로딩
# 9B 모델을 원하시면 "google/gemma-2-9b-it"로 변경하고 4-bit 설정을 추가해야 합니다.
llm_id = "google/gemma-2-2b-it"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_id, token=clean_token)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=clean_token
)
print("✅ 시스템 초기화 및 Gemma-2 모델 장착 완료!\n")

# ==========================================
# 2단계: 로봇 카메라 이미지 로드
# ==========================================
image_path = "/content/image(7).png" # 👈 실제 경로 확인!
image = Image.open(image_path).convert("RGB")

total_start_time = time.time() # 🏁 전체 프로세스 시작 시간 기록

# ==========================================
# 3단계: 시각 인지 (Florence-2) - 현장 스캔
# ==========================================
print("🔍 [현장 분석] Florence-2가 시각 데이터를 분석 중...")
florence_start = time.time()

task_prompt = "<MORE_DETAILED_CAPTION>"
inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(device)

with torch.no_grad():
    generated_ids = vision_model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=image.size)
florence_description = parsed_answer[task_prompt]

florence_end = time.time()
florence_duration = florence_end - florence_start

print(f"👀 [시각 센서 데이터 ({florence_duration:.2f}초)]:")
print(florence_description)
print("-" * 50)

# ==========================================
# 4단계: 중앙 통제 AI 판단 (Gemma-2) - 논리 추론
# ==========================================
print("🤔 [위험도 평가] 중앙 통제 AI(Gemma-2)가 분석을 시작합니다...\n")
gemma_start = time.time()

user_question = """1. 이 물체가 종합적으로 무엇일 가능성이 높은지 추론하세요.
2. 현재 상황의 위험도를 평가하세요.
3. 로봇이 즉시 취해야 할 행동 지침을 명확하게 명령하세요."""

# Gemma 2는 프롬프트 형식이 약간 엄격할 수 있으므로, 시스템 지시를 user 메시지에 통합하는 것도 좋은 전략입니다.
messages = [
    {"role": "user", "content": f"[지시사항] 당신은 위험 환경을 탐색하는 모바일 로봇(AMR)의 중앙 통제 인공지능입니다. 시각 센서가 보내온 텍스트를 분석하여 반드시 '한국어'로 논리적으로 답변하세요.\n\n[시각 센서 데이터]:\n{florence_description}\n\n[질문]:\n{user_question}"}
]

prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
llm_inputs = llm_tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = llm_model.generate(
        input_ids=llm_inputs["input_ids"],
        attention_mask=llm_inputs["attention_mask"],
        max_new_tokens=1024,
        temperature=0.3,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id
    )

input_length = llm_inputs["input_ids"].shape[1]
response = llm_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

gemma_end = time.time()
gemma_duration = gemma_end - gemma_start
total_end_time = time.time()

# ==========================================
# 5단계: 최종 리포트 출력
# ==========================================
print("========================================")
print("🚨 AMR 중앙 통제 AI (Gemma-2 2B) 최종 명령 🚨")
print("========================================")
print(response)
print("========================================")
print(f"⏱ [성능 리포트]")
print(f"- 시각 인지 (Florence-2): {florence_duration:.2f}초")
print(f"- 논리 추론 (Gemma-2 2B): {gemma_duration:.2f}초")
print(f"- 전체 처리 시간: {total_end_time - total_start_time:.2f}초")
print("========================================")
# 현재 GPU 메모리 사용량을 MB 단위로 출력
print(f"📊 실시간 VRAM 점유량: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")